# Project 4: **Build a Deep Research System**
Welcome to project 4! For this project, we shift our focus from tool use and agents to *reasoning* models. You will practice state‑of‑the‑art inference‑time scaling methods such as *Chain‑of‑Thought* prompting and *Tree‑of‑Thoughts*, and briefly explore high-level concepts of training reasoning models using techniques like **STaR**.


Finally, you will put everything together to build a *deep research agent* that can browse the web, reason over what it finds, and give structured answers.

## Learning Objectives  
* Apply common inference‑time scaling methods: **zero‑shot / few‑shot CoT, self‑consistency, sequential revision, tree‑of‑thoughts**  
* Gain intuition for **training** reasoning‑capable models following **STaR** approach 
* Build a minimal **deep‑research agent** that combines step‑by‑step reasoning with live web search   
* Practice extending deep-search to a multi-agent system 

## Roadmap  
0. Environment setup  
1. Inference‑time scaling  
  1.1 Few‑shot.   
  1.2 Zero‑shot CoT.   
  1.3 Self‑consistency.   
  1.4 Sequential revisions.     
  1.5 Tree‑of‑Thought (ToT)
2. Training reasoning models and inspecting deepseek-r1 
3. Deep-research agent  
4. (Optional) Multi-agent deep-research

# 0- Environment setup

### Step 1: Create your environment and install dependencies 
Before we start coding, you need a reproducible setup. Open a terminal in the same directory as this notebook, and use Conda or uv to install the project dependencies.

#### Option 1: Conda
```bash
# Create and activate the conda environment
conda env create -f environment.yaml && conda activate deep_research
```

#### Option 2: uv (faster)
If you prefer [uv](https://docs.astral.sh/uv/) over Conda:

```bash
# Install uv (skip if already installed)
curl -LsSf https://astral.sh/uv/install.sh | sh

# Create a virtual environment and install dependencies
uv venv .venv --python 3.11 && source .venv/bin/activate
uv pip install -r requirements.txt
```

### Step 2: Register this environment as a Jupyter kernel
```bash
python -m ipykernel install --user --name=deep_research --display-name "deep_research"
```
Now open your notebook and switch to the `deep_research` kernel (Kernel → Change Kernel).

### Step 3: Setup and run Ollama serve

In this project we use the `llama3.2:3b`, `qwen2.5:3b-instruct` and `deepseek-r1:1.5b` models. You can try other smaller or larger reasoning LLMs such as `phi4-mini` to compare performance. Explore available models here: https://ollama.com/library.

Open terminal and run ollama:
```bash
ollama serve
```
Then open another terminal and pull required models: 
```bash
ollama pull llama3.2:3b
ollama pull deepseek-r1:1.5b
ollama pull qwen2.5:3b-instruct
# Additional small reasoning models to compare
# ollama pull phi4-mini
```

---  
# 1‑ Inference‑time scaling

Inference-time scaling refers to techniques that make an existing model reason better without retraining it. Instead of changing the model’s weights, we achieve reasoning capability by adjusting how we prompt, sample, or aggregate LLM's outputs.

In this section, we’ll explore several inference-time strategies that improve reasoning quality using a non-reasoning base model. You will experiment with and compare methods such as:

- Few-shot Chain-of-Thought (CoT)
- Zero-shot CoT
- Self-consistency
- Sequential revision
- Tree-of-Thoughts (ToT)

### 1.1: Few-Shot CoT

Few-shot prompting provides examples before asking a new question. The model learns from the pattern and applies it to new inputs.

We'll explore this with two models to understand how few-shot interacts with model capabilities:

1. **GPT-2** (no instruction tuning): Doesn't reason by default. We'll see if few-shot examples can elicit reasoning.
2. **Llama 3.2** (instruction-tuned): Already reasons naturally. We'll use few-shot to control the output format.

#### GPT-2: Can few-shot examples elicit reasoning?

GPT-2 is a base language model that just predicts the next token. It wasn't trained to follow instructions or reason step-by-step. Let's see what happens with and without few-shot examples.

In [5]:
import os
import torch
from transformers import pipeline

question = "A rectangle has a perimeter of 36 cm. If the length is twice the width, what is the area?"

# Step 1: Load GPT-2 text-generation from huggingface (https://huggingface.co/docs/transformers/en/model_doc/gpt2)
# Step 2: Write 1–2 few-shot reasoning examples (short, explicit steps + final answer in your own unique format)
# Step 3: Append a new test question after the examples to form one prompt string
# Step 4: Generate outputs with and without fewshot prompt and compare the difference.

os.environ["PYTORCH_ENABLE_MPS_FALLABCK"] = "1"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
if device.type == "mps":
    torch.mps.empty_cache()

# Load GPT-2 from HuggingFace
generator = pipeline(task="text-generation", model="openai-community/gpt2", dtype=torch.float16, device=device)

# Few-shot reasoning examples
fewshot_examples = """
Q: There are 5 apples in a basket. If you take away 3 apples, how many are left?
Let's think step by step.
Step 1: There are 5 apples.
Step 2: I take away 3 apples.
Step 3: 5 - 3 = 2 apples remain.
Therefore, the answer is 2

Q: If a train travels 60 miles in 2 hours, what is its average speed?
Let's think step by step.
Step 1: The train travels 60 miles in 2 hours.
Step 2: Speed = distance / time = 60 / 2 = 30 miles per hour.
Therefore, the answer is 30 miles per hour
"""

# Zero shot question
# Zero-shot prompt (no few-shot examples)
zeroshot_prompt = f"Q: {question}\nA:"

# New test question
test_question = f"Q: {question}\nA:"

# Prompt with few-shot
fewshot_prompt = fewshot_examples + "\n" + test_question

print("=== GPT-2 without few-shot ===")
print(generator(question, max_new_tokens=300, do_sample=True, temperature=0.8)[0]["generated_text"])
print("\n=== GPT-2 with few-shot examples ===")
print(generator(fewshot_prompt, max_new_tokens=300, do_sample=True, temperature=0.8)[0]["generated_text"])


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== GPT-2 without few-shot ===


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=300) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A rectangle has a perimeter of 36 cm. If the length is twice the width, what is the area?

The width in the rectangle is to be measured from the corner of the rectangle to the corner of the edge, or from the corner of the edges to the corner of the rectangle.

The rectangle has a radius of 36 cm.

The rectangle has a radius of 44 cm.

The rectangle has a radius of 51.5 cm.

The rectangle has a radius of 53.5 cm.

The rectangle has a radius of 54.5 cm.

The rectangle has a radius of 56.5 cm. The rectangle has a radius of 56.5 cm. The rectangle has a radius of 55.5 cm. The rectangle has a radius of 56.5 cm. The rectangle has a radius of 57.5 cm. The rectangle has a radius of 59.5 cm. The rectangle has a radius of 59.5 cm.

The rectangle has a radius of 60.5 cm.

The rectangle has a radius of 67.5 cm.

The rectangle has a radius of 70 cm.

The rectangle has a radius of 73 cm.

The rectangle has a radius of 74 cm. The rectangle has a radius of 75 cm. The rectangle has a radius of 76.5 cm. 

#### Llama 3.2: Using few-shot to control output format

Unlike GPT-2, Llama 3.2 is instruction-tuned and already produces reasoning traces by default. So what's the point of few-shot examples?

**The power of few-shot with instruction-tuned models is controlling the output format.** We can make the model follow a specific structure like `[GIVEN]/[FIND]/[SOLVE]/[ANSWER]` that it wouldn't use naturally.

In [7]:
from openai import OpenAI

question = "A rectangle has a perimeter of 36 cm. If the length is twice the width, what is the area?"

# Step 1: Create your Ollama client
# Step 2: Write a few examples showing reasoning steps
# Step 3: Concatenate examples + new question into a single prompt
# Step 3: Call your Ollama or OpenAI client to get a response from llama3.2:3b # e.g., client.chat.completions.create(...)
# Step 5: Print the final answer with and without few shot examples and compare them.

client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "llama3.2:3b"

# Use the same few-shot examples from earlier
fewshot_examples = """
Q: There are 5 apples in a basket. If you take away 3 apples, how many are left?
Let's think step by step.
Step 1: There are 5 apples.
Step 2: I take away 3 apples.
Step 3: 5 - 3 = 2 apples remain.
Final Answer: 2

Q: If a train travels 60 miles in 2 hours, what is its average speed?
Let's think step by step.
Step 1: The train travels 60 miles in 2 hours.
Step 2: Speed = distance / time = 60 / 2 = 30 miles per hour.
Final Answer: 30 miles per hour
"""

test_question = f"Q: {question}\nLet's think step by step.\n"
fewshot_prompt = fewshot_examples + "\n" + test_question

# Llama3.2 zero-shot
print("=== Llama3.2 Zero-shot ===")
resp_zero = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": test_question}],
    temperature=0.3,
)
print(resp_zero.choices[0].message.content.strip())

# Llama3.2 with few-shot examples
print("\n=== Llama3.2 Few-shot ===")
resp_fewshot = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": fewshot_prompt}],
    temperature=0.3,
)
print(resp_fewshot.choices[0].message.content.strip())

=== Llama3.2 Zero-shot ===
To find the area of the rectangle, we need to know its length and width. We are given that the length is twice the width.

Let's call the width "w" (in cm). Since the length is twice the width, the length can be represented as 2w.

The perimeter of a rectangle is the sum of all its sides. In this case, it's the sum of the two lengths and the two widths:

Perimeter = 2(length) + 2(width)
36 = 2(2w) + 2(w)

Simplify the equation:
36 = 4w + 2w
36 = 6w

Now, divide both sides by 6 to solve for w:
w = 36 / 6
w = 6 cm (width)

Since the length is twice the width, we can find the length:
Length = 2w
= 2(6)
= 12 cm

Now that we have the length and width, we can calculate the area:

Area = Length × Width
= 12 × 6
= 72 square centimeters (cm²)

Therefore, the area of the rectangle is 72 square centimeters.

=== Llama3.2 Few-shot ===
To solve this problem, let's break it down step by step.

Step 1: The perimeter of the rectangle is given as 36 cm. The formula for the pe

### 1.2: Zero‑Shot Chain‑of‑Thought
Zero-shot CoT encourages the model to reason without examples by adding a short cue such as “Let’s think step by step.” This simple phrase often activates the model’s latent reasoning ability even when no demonstrations are provided. It serves as a baseline to compare with few-shot and other inference-time scaling methods.

In [8]:
from openai import OpenAI

# Step 1: Create your Ollama client
# Step 2: Write a question and a zero-shot CoT cue (e.g., "Let's think step by step.")
# Step 3: Build a single prompt string that includes brief role guidance plus the question
# Step 3: Call your Ollama or OpenAI client to get a response from llama3.2:3b  # e.g., client.chat.completions.create(...)
# Step 4: Print the chain and the final answer

client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)
MODEL = "llama3.2:3b"

question = "If a factory produces 120 widgets in 8 hours, how many widgets does it produce per hour?"
role_guidance = "You are a helpful math tutor. Answer the following question with step-by-step reasoning."
prompt = f"{role_guidance}\nQ: {question}\nLet's think step by step."

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.3,
)
print(response.choices[0].message.content.strip())



To find out how many widgets the factory produces per hour, we need to divide the total number of widgets produced (120) by the total number of hours it took to produce them (8).

Here are the steps:

1. Write down the information given in the problem:
Total widgets = 120
Total hours = 8

2. Identify what we want to find: The rate at which the factory produces widgets per hour.

3. Choose a unit of measurement for the rate: Let's use widgets per hour (wph).

4. Set up a division equation using the information from step 1:
Widgets per hour (wph) = Total widgets ÷ Total hours
= 120 ÷ 8

5. Perform the division operation:
120 ÷ 8 = 15

6. Write down the answer:
The factory produces 15 widgets per hour.

That's it!


### 1.3 Self‑Consistency
Self-consistency enhances reasoning accuracy by sampling multiple independent reasoning paths for the same question instead of relying on a single deterministic answer. Each run may follow a slightly different logical chain, and the diversity helps correct individual mistakes. After generating several reasoning traces, you then aggregate the final answers using majority voting.

This approach is especially useful when tasks involve multi-step reasoning or arithmetic, where single-path outputs may be incorrect.

In [9]:
from openai import OpenAI
import re, collections

client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "llama3.2:3b"


def cot_answer(question, temperature=1.2):
    # Generate a step-by-step reasoning chain for the given question and extract the final answer.
    # Use a Chain-of-Thought (CoT) prompt, call the model, and extract the final answer from its output.
    role_guidance = 'You are a helpful tutor. Answer the question step by step. End your answer with exactly: "Therefore, the answer is [x]" where [x] is just the final answer.'
    prompt = f"{role_guidance}\nQ: {question}\nLet's think step by step."
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    output = response.choices[0].message.content.strip()
    # Try to extract the final answer after 'Answer:'
    match = re.search(r"Answer:\s*([^\n]*)", output, re.IGNORECASE)
    if match:
        answer_raw = match.group(1)
        # Remove leading/trailing dots and spaces
        answer_clean = answer_raw.strip(" .\n")
        match = re.match(r"(.*)", answer_clean)
    final_answer = match.group(1).strip() if match else output
    print(final_answer)
    return final_answer, output


def self_consistent(question, n=5):
    # Run multiple reasoning chains and select the most frequent final answer by majority voting.
    answers = []
    traces = []
    for _ in range(n):
        ans, trace = cot_answer(question)
        answers.append(ans)
        traces.append(trace)
    counter = collections.Counter(answers)
    winner, _ = counter.most_common(1)[0]  # most frequent answer
    return winner, counter, traces

question = "What is the square root of 144?"
winner, counter, traces = self_consistent(question, n=5)

print("Votes:", counter)
print("Chosen answer:", winner)

To find the square root of 144, we'll break it down step by step:

Step 1: Start by thinking about numbers that you know are squared to produce the number 144. This might help us identify the factors.

Step 2: We can begin testing numbers by squaring them. Let's start with perfect squares like 11 (since 9^2 is already around but not exactly, and isn't it funny what year the world started with?)) which are close, like 12 or something.

11² = 121 => Not quite there we do it more simple.

Try 13: 
 
(13)² = 169

Step 3:  Our next guess will be one number higher.

(14)² = 196

Since we want an answer less than that we take the second lowest guess from (12).

 
We see this:
 (11). So it's actually more low.


Check whether square of the guessed 12 is not too much but slightly above.

Let's do a simple square:
7² = 49. Our desired amount squared needs to have an upper limit value so check its square and make sure the lower bound fits perfectly:

We find
12²=144= (12)⁵. This makes sense when 

### 1.4: Sequential Revision

Sequential revision iteratively improves an answer by generating a first draft, critiquing it, and producing revised drafts that condition on prior answers. Each round should be short and focused, so improvements accumulate without drifting from the question.

In [10]:
from openai import OpenAI

client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "llama3.2:3b"


def sequential_revision(question: str, max_steps: int = 3) -> str:
    # Generate an initial draft answer, then iteratively refine it by conditioning each revision on the previous one.
    # Step 1: Ask the model to produce the first draft for the given question
    # Step 2: Loop for max_steps-1 times, each time feeding the last draft back to the model with a request to revise
    # Step 3: Print each draft to observe how the answer evolves
    # Step 4: Return the final improved draft
    # Step 1: Generate the initial draft
    drafts = []
    # Initial prompt
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": f"Answer the following question as best you can:\n\nQuestion: {question}"}
        ],
        temperature=0.7,
        max_tokens=256,
    )
    draft = response.choices[0].message.content.strip()
    drafts.append(draft)
    print(f"Draft 1:\n{draft}\n{'-'*40}")

    # Step 2: Iteratively revise the draft
    for step in range(1, max_steps):
        prompt = (
            f"The following is an answer draft to the question.\n"
            f"Question: {question}\n"
            f"Draft:\n{drafts[-1]}\n\n"
            f"Revise and improve the answer to be more accurate, clear, and complete, without making up information or losing focus on the original question."
        )
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=256,
        )
        draft = response.choices[0].message.content.strip()
        drafts.append(draft)
        print(f"Draft {step+1}:\n{draft}\n{'-'*40}")

    # Step 4: Return the final improved draft
    return drafts[-1]

# Step 1: Define a question that benefits from multi-step reasoning
# Step 2: Call sequential_revision(question, max_steps)
# Step 3: Print the final output
question = "What are the main causes of the fall of the Roman Empire?"
final_answer = sequential_revision(question, max_steps=3)

Draft 1:
The fall of the Roman Empire is a complex and multifaceted topic, and there is no single answer to what caused its decline. However, historians have identified several key factors that contributed to the empire's collapse. Here are some of the most significant causes:

1. **Internal Decay and Corruption**: The Roman Empire was plagued by corruption, mismanagement, and internal decay. As the empire grew in size and complexity, it became increasingly difficult to maintain unity and discipline among its vast territories. Corruption was rampant, with officials embezzling funds, bribing officials, and engaging in other forms of malfeasance.
2. **External Pressures**: The Roman Empire faced numerous external threats from neighboring tribes and states, including the Huns, Goths, Vandals, and Visigoths. These barbarian groups frequently raided Roman territories, pillaged cities, and eventually broke through the empire's defenses to wreak havoc on its populations.
3. **Military Overext

### 1.5 Tree-of-Thoughts

Tree-of-Thoughts (ToT) reframes reasoning as a search problem. Instead of generating one linear chain of thoughts, the model:
1. Generates multiple candidate "thoughts" at each step
2. Evaluates how promising each thought is
3. Expands only the best candidates (beam search)
4. Backtracks if needed

This mirrors how humans solve hard problems: brainstorm options, evaluate them, pursue the best, and backtrack when stuck.

#### Example 1 (Optional): Word Ladder (Algorithmic ToT)

This example shows ToT as pure beam search without LLM calls. Each "thought" is a candidate word that differs by one letter. We score by edit distance to goal and keep the best candidates.

This demonstrates the **core algorithm** behind ToT: expand, score, prune.

In [12]:
###### Word Ladder Puzzle ##########

def neighbors(word, vocabulary):
    # Generate all valid one-letter mutations of 'word' that exist in 'vocabulary' and return them.
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    neighbors_set = set()
    word = word.lower()
    for i in range(len(word)):
        for c in alphabet:
            if c != word[i]:
                candidate = word[:i] + c + word[i+1:]
                if candidate in vocabulary:
                    neighbors_set.add(candidate)
    return list(neighbors_set)


def tree_of_thought(start, goal, vocab, max_depth=5, beam_width=4):
    # Search over partial thoughts (paths) using a small beam.
    # Step 1: Initialize the frontier with a single path [start]
    # Step 2: For each depth, expand each path by one neighbor from 'neighbors'
    # Step 3: Score paths by edit distance between last word and 'goal' (smaller is better)
    # Step 4: Keep the top 'beam_width' paths and stop early if any reaches 'goal'
    # Step 5: Return the best goal-reaching path or None
    frontier = [[start]]
    best_path = None

    for depth in range(max_depth):
        new_frontier = []
        for path in frontier:
            last_word = path[-1]
            if last_word == goal:
                # Found goal, early stopping
                return path
            nbrs = neighbors(last_word, vocab)  # Each neighbor is a possible extension
            for nbr in nbrs:
                if nbr not in path:  # Avoid cycles
                    new_frontier.append(path + [nbr])
        if not new_frontier:
            break
        # Score paths by edit distance between last word and goal (lower is better)
        scored = sorted(
            new_frontier,
            key=lambda p: sum(a != b for a, b in zip(p[-1], goal))
        )
        # Prune to the top beam_width paths
        frontier = scored[:beam_width]
        # Check if any path ends in goal
        for p in frontier:
            if p[-1] == goal:
                return p
    # After search, return the best path ending with goal, or None
    for p in frontier:
        if p[-1] == goal:
            return p
    return None


vocab = {"hit","dot","cog","log","dog","lot","lit","hot"}
print(tree_of_thought("hit", "cog", vocab))


['hit', 'hot', 'lot', 'log', 'cog']


#### Example 2: Generic ToT for Open-Ended Problems

For open-ended problems without verifiable answers, we can still apply ToT by having the LLM both propose and evaluate thoughts.

In [16]:
###### Generic ToT Search ##########

import re
from openai import OpenAI

client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "llama3.2:3b"

def propose_thoughts(question, state, k=2):
    # Propose up to k next “thoughts” that extend the current partial solution/state.
    # Steps: build a short prompt with problem + current state; call your client. Then return a list of stripped strings.
    prompt = (
        f"You are brainstorming next steps for the following problem.\n"
        f"Question: {question}\n"
        f"Current partial solution/plan: {state}\n"
        f"\nList up to {k} next possible steps or thoughts to extend/improve the plan. "
        f"Return each thought on its own line without numbering or extra text."
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=256,
        n=1,
    )
    thoughts = [s.strip() for s in response.choices[0].message.content.strip().split('\n') if s.strip()]
    return thoughts[:k]


def score_state(question, state):
    # Score how promising a partial solution is on a 1–10 scale (higher is better).
    # Steps: build a rating prompt; call the model; parse the first integer 1–10;
    prompt = (
        f"You are evaluating the quality and promise of a partial plan for the following question:\n"
        f"Question: {question}\n"
        f"Partial plan so far: {state}\n"
        f"\nOn a scale of 1 to 10 (10 = extremely promising, 1 = poor), how promising is this partial plan? "
        f"Respond with only a single integer from 1 to 10."
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=8,
    )
    match = re.search(r"\b([1-9]|10)\b", response.choices[0].message.content)
    score = int(match.group(1)) if match else 1
    return score


def tree_of_thoughts(question, depth=2, width=2):
    # Run a tiny ToT search: expand states with propose_thoughts, score with score_state, keep top-k at each depth.
    # Steps: initialize frontier=[("", 0)]; for each depth, expand each state with k=width thoughts; score each; sort by score desc; keep top 'width'; return best state and score.
    frontier = [("", 0)]  # Each state is (plan_so_far, score)
    best_state = ("", 0)

    for d in range(depth):
        candidates = []
        for state, _ in frontier:
            thoughts = propose_thoughts(question, state, k=width)
            for t in thoughts:
                new_state = state + ("\n" if state else "") + t
                score = score_state(question, new_state)
                print(f"New state: {new_state}, score: {score}, depth: {d}, width: {width}, thoughts: {t}")
                candidates.append((new_state, score))
        # Sort candidates by score descending, take top 'width'
        candidates.sort(key=lambda x: x[1], reverse=True)
        frontier = candidates[:width]
        if frontier:
            best_candidate = frontier[0]
            if best_candidate[1] > best_state[1]:
                best_state = best_candidate

    return best_state


question = "Design a plan for a weekend science workshop for 12-year-olds."
solution, score = tree_of_thoughts(question)

print(f"Best solution (score {score}):\n{solution}")

New state: Identify specific STEM topics that will engage 12-year-olds, such as robotics, coding, environmental science, or physics experiments., score: 6, depth: 0, width: 2, thoughts: Identify specific STEM topics that will engage 12-year-olds, such as robotics, coding, environmental science, or physics experiments.
New state: Partner with local schools or community centers to secure a venue and gather volunteers to assist with the workshop, potentially reducing costs and increasing the overall experience for participants., score: 6, depth: 0, width: 2, thoughts: Partner with local schools or community centers to secure a venue and gather volunteers to assist with the workshop, potentially reducing costs and increasing the overall experience for participants.
New state: Identify specific STEM topics that will engage 12-year-olds, such as robotics, coding, environmental science, or physics experiments.
Develop a list of hands-on activities and projects that can be completed within a s

---  
# 2‑ Training Models for Reasoning

### 2.1: CoT Training
Chain-of-Thought (CoT) training conditions the model on explicit rationales during fine-tuning. Instead of teaching the model to output only the final answer, we train on (question, rationale, answer) so the model learns to internalize multi-step reasoning patterns. A practical recipe is STaR (Self-Taught Reasoner), which uses a stronger teacher model to bootstrap rationales that a smaller student can learn from.

For tasks that require multi-hop reasoning, models fine-tuned on rationales often achieve higher accuracy and are more stable at inference time than models trained on direct answers only. 

Training a full language model is beyond the scope of this notebook, but here is the high-level workflow followed by a short pseudocode:
- Collect questions: Prepare a dataset of questions and correct answers.
- Generate rationales: Use a strong LLM to produce step-by-step reasoning ending with the correct answer.
- Filter and clean: Discard incorrect or low-quality rationales.
- Prepare training data: Format triples (question, rationale, answer) for supervised fine-tuning.
- Fine-tune: Fine-tune the LLM on rationales.
- Iterate: Refine prompts, improve data quality, and retrain for stronger reasoning.

In [17]:
# Pseudocode (STaR loop)
# for round in 1 ... iters:
    # STEP 1: self-generate reasoning (teacher creates rationale + answer)
    # STEP 2: keep only correct, high-quality traces
    # STEP 3: fine-tune student on (question, rationale, answer) data

### 2.2: ORM vs PRM + RL
Training a Reward Model (RM) allows large language models to be improved through reinforcement learning (RL). Instead of fine-tuning directly on examples, we train a separate model that can score or rank model outputs, and use those scores as feedback signals to refine the policy model.

Two main reward modeling approaches are ORM (predicts a scalar reward for the final answer) and PRM (evaluates the reasoning steps instead of just the outcome)



| Approach | Typical loss | When to use |
|-----------|-------------|-------------|
|*Outcome Reward Model* | Predict scalar reward | Easy to collect training data using verifiers |
|*Process Reward Model* | Predict rewards per step | Difficult to collect training data but more accurate |
| *RLHF* | Use RM as reward in **RL** fine‑tuning | Aligns policy with human signals | Aligns model policy with human or synthetic preferences




In [28]:
# for round = 1 ... iters:
    # STEP 1:  Generate reasoning
        # sample a minibatch of questions
        # policy roll‑out (actions + log‑probs)
    # STEP 2:  Score the trajectory
        # ORM: scalar reward for the final answer / PRM: scalar reward for the thought process
    # STEP 3:  Reinforce the policy (PPO)

### 2.3 Inspect a reasoning model

Now that we've discussed how reasoning models are trained, let's see one in action. We'll use **DeepSeek-R1**, a reasoning model that produces explicit *thinking tokens* before giving its final answer. The model wraps its internal chain-of-thought inside `<think>...</think>` tags, followed by a clean final response.

In the cell below we send a question to DeepSeek-R1 and parse the output to separate:
- **Thinking tokens** — the model's internal reasoning process (hidden from the end user in production).
- **Final answer** — the polished response the user actually sees.

We use `deepseek-r1:1.5b` here for speed. You can switch to `deepseek-r1:8b` for higher-quality reasoning, but it will take longer to run. Pull whichever variant you want to try:

In [11]:
import re
from openai import OpenAI

# Step 1: Create OpenAI client and set your DeepSeek Model
# Step 2: Write a math question
# Step 3: Call your model
# Step 4: Inspect the output. Separate thinking and final answer sections and print them.
# Step 1: Create OpenAI client and set your DeepSeek Model
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # You can put anything, unused by Ollama.
)
MODEL = "deepseek-r1:1.5b"

# Step 2: Write a math question
question = "What is the sum of the first 10 positive integers?"

# Step 3: Call your model
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
    temperature=0.2,
    max_tokens=256,
)
output = response.choices[0].message.content.strip()

# Step 4: Inspect the output. Separate thinking and final answer sections and print them.
# DeepSeek reasoning is in <think>...</think>    """Search the web using DuckDuckGo and return top results with titles and URLs."""
thinking = re.search(r"<think>(.*?)</think>", output, re.DOTALL)
think_text = thinking.group(1).strip() if thinking else ""
final_answer = re.sub(r"<think>.*?</think>", "", output, flags=re.DOTALL).strip()

print("=== Thinking ===")
print(think_text)
print("\n=== Final Answer ===")
print(final_answer)

=== Thinking ===


=== Final Answer ===
To find the sum of the first 10 positive integers, we can use Gauss's method for quickly calculating the sum of an arithmetic series.

**Step-by-Step Solution:**

1. **Identify the Series:**
   
   The first 10 positive integers are:
   \[
   1, 2, 3, 4, 5, 6, 7, 8, 9, 10
   \]

2. **Pair the Numbers:**
   
   Pair the numbers from the start and end towards the center:
   \[
   (1 + 10),\ (2 + 9),\ (3 + 8),\ (4 + 7),\ (5 + 6)
   \]


# 3‑ A Deep Research Agent

A deep-research agent pairs a reasoning model with external tools for web search and retrieval. We will follow the ReAct pattern: the model writes short thoughts, decides when to call tools, reads observations, and continues reasoning until it can answer or reaches a step limit.

We now combine a **search tool** with an LLM in a multi-step setup. We follow the *ReAct* pattern (reason → tool → observation):

1. The model reasons and decides to use tools
2. The agent searches and feeds condensed snippets back as context
3. Iterate until the model answers or hits a step limit

We use `create_agent` from `langchain.agents`, which builds a ReAct-style agent graph. Note: the agent model must support **tool calling** (e.g., `llama3.2:3b`). Models like `deepseek-r1` are reasoning models that do not support native tool calling and cannot be used directly as the agent LLM. We can stick to the `llama3.2:3b` or `qwen2.5:3b-instruct` for this section.

In [12]:
from ddgs import DDGS
from langchain_core.tools import tool


@tool
def ddg_search(query: str, k: int = 5) -> str:
    """Search the web using DuckDuckGo and return top results with titles and URLs."""
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=k)
        entries = [f"- {r['title']} - {r['href']}" for r in results]
    return "\n".join(entries) if entries else "No results found."

In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

MODEL = "qwen2.5:3b-instruct"
question = "What are the best resources to learn machine learning in 2025?"

# Step 1: Initialize the LLM via ChatOllama (must support tool calling)
llm = ChatOllama(model=MODEL, temperature=0.7)

# Step 2: Build a tool-calling agent with DuckDuckGo search
tools = [ddg_search]
web_agent = create_agent(llm, tools)

# Step 3: Ask a query and let the agent search + reason to produce an answer
messages = {
    "messages": [
        {"role": "user", "content": question}
    ]
}
result = web_agent.invoke(messages)
print(result)
result["messages"][-1].content

Impersonate 'chrome_128' does not exist, using 'random'


{'messages': [HumanMessage(content='What are the best resources to learn machine learning in 2025?', additional_kwargs={}, response_metadata={}, id='164dbe1f-44fe-4e70-b4d5-51e1468f79cc'), AIMessage(content="To provide you with the most relevant information, I would need to search the web. Let's try searching for educational resources related to machine learning.\n", additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b-instruct', 'created_at': '2026-03-23T17:49:38.137816Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3654182458, 'load_duration': 1785312375, 'prompt_eval_count': 178, 'prompt_eval_duration': 470375750, 'eval_count': 60, 'eval_duration': 1336873996, 'logprobs': None, 'model_name': 'qwen2.5:3b-instruct', 'model_provider': 'ollama'}, id='lc_run--019d1bd0-fd92-7243-9313-8899aefd34b2-0', tool_calls=[{'name': 'ddg_search', 'args': {'query': 'best resources to learn machine learning 2025'}, 'id': '1c351dba-586e-4ec6-9532-25cba42db20a', 'type': 'tool_call'}],

"Here are some resources related to learning machine learning in the year 2025:\n\n1. Machine Learning Training - AI Training Courses: [Link](https://www.bing.com/aclick?ld=e8ZlCn_QSwha8jYunWt_NsYzVUCUwPo7tAez3DSrc9T5zdAEy_TeAttrXTunXEZJMc8V_q6PQ6dah-AHmongbw9_ehHm5b4gIiPklBlPhDUxpSEiAm6dR-TILqLt5re4HBv3iG74d9f-LwmxvKAaIKf4UeQGuFhAt-EUyPaWZdjqEoz_bffaUE6KrVnJemVMZJMF42w9qAEy-Kotrmz639YnUe7MM&u=aHR0cHMlM2ElMmYlMmZwaXhlbC5ldmVyZXN0dGVjaC5uZXQlMmY0NDIyJTJmY3ElM2Zldl9zaWQlM2QxMCUyNmV2X2xuJTNkbWFjaGluZSUyNTIwbGVhcm5pbmclMjUyMGNvdXJzZSUyNTIwb25saW5lJTI2ZXZfbHR4JTNkJTI2ZXZfbHglM2Rrd2QtNzE2MDY1NDA2NjkzMjUlM2Fsb2MtMzglMjZldl9zcGQlM2QxMCUyNmV2X2lkJTNkMTE0NTY5Mjk2Mzk5MjU4NCUyNmV2X2R2YyUzZGMlMjZldl9waHklM2QlMjZldl9jeCUzZDQ4NzQ0MTg3NiUyNmV2X2F4JTNkMTE0NTY5Mjk2Mzk5MjU4NCUyNmV2X2lkJTI1MjZzY19jaGFubmVsJTI1M0RwcyUyNTI2c19rd2NpZCUyNTNEQUwhNDQyMiExMCE3MTYwNTk3OTUzMzYzOSEhISE3MTYwNjU0MDY2OTMyNSEhNDg3NDQxODc2ITExNDU2OTI5NjM5OTI1ODQlMjUyNmVmX2lkJTI1M0QyNDkwYWVhZjkxMzYxMzUyOWMxM2M3NmVlYzAyMDYzYSUyNTNBRyUyNTN

# 4- (Optional) Multi-Agent Deep Research

Instead of a single agent, we can design multiple collaborating agents that work in parallel:

1. **Planner**: Analyzes the query and breaks it into sub-questions
2. **Researchers**: Run in parallel, each searching and summarizing findings for one sub-question  
3. **Synthesizer**: Combines all research into a coherent final report

This setup improves coverage and speed by parallelizing the research phase.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from ddgs import DDGS

client = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")
MODEL = "llama3.2:3b"


def plan_research(query: str) -> list[str]:
    """Planner agent: breaks query into sub-questions."""
    # Prompt the LLM to decompose the query into 1-5 focused sub-questions.
    # The prompt should instruct the model to return only sub-questions, one per line.
    # Parse the response into a list of stripped strings and return
    # The planner asks the LLM to break the query into 1-5 focused sub-questions,
    # returning only the sub-questions, one per line.
    planner_prompt = f"""You are a research planner. Break the following question into 1 to 5 focused sub-questions, "
        each covering a distinct aspect. 
        - Simple factual queries: 1 sub-question
        - Mpderate topics: 3 sub-questions
        - Complex topics needing multiple perspectives: 5 sub-questions

        Query: {query}
        
        Return ONLY sub-questions, one per line, no numbering or explanation or bullet points."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": planner_prompt}
        ],
        max_tokens=200,
        temperature=0.3,
        stop=None,
    )
    sub_questions = [line.strip() for line in response.choices[0].message.content.strip().split("\n") if line.strip()]
    return sub_questions[:5] # Limit to 5 sub-questions


def search_and_summarize(sub_question: str) -> dict:
    """Researcher agent: searches web and summarizes findings for one sub-question."""
    # Step 1: Use DDGS to search the web for the sub-question (max_results=3)
    # Step 2: Join the result snippets into a single string
    # Step 3: Prompt the LLM to write a concise summary based on the snippets
    # Step 4: Return a dict with keys "question" and "summary"
    # Use the DDGS tool (already imported above) to search the web
    snippets = []
    search_results = None
    with DDGS() as ddgs:
        search_results = [hit["body"] for hit in ddgs.text(sub_question, max_results=3)]

    # Aggregate all snippets/bodies from search results
    snippets = "\n".join(search_results)


    # Use the LLM tool defined above (client) to generate a summary based on the search results
    summary_prompt = """
    Given the following web search results, summarize concisely (3-6 sentences) to answer the research question.
    
    Research question: {sub_question}
    Search results: {snippets}
    Summary:
    """
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": summary_prompt}
        ],
        max_tokens=300,
        temperature=0.3,
        stop=None,
    )
    summary = response.choices[0].message.content.strip()

    # Return the result as a dictionary
    return {"question": sub_question, "summary": summary}


def synthesize_report(query: str, findings: list[dict]) -> str:
    """Synthesizer agent: combines all findings into a coherent report."""
    # Step 1: Format the findings list into a readable text block (e.g., "### sub-question\nsummary" per finding)
    # Step 2: Prompt the LLM to combine them into a well-structured markdown report that answers the original query
    # Step 3: Return the report string
    """
    YOUR CODE HERE (~10 lines of code)
    """
    pass


def deep_research(query: str) -> str:
    """Run the full multi-agent deep research pipeline."""
    # Step 1: Call plan_research to break the query into sub-questions and print them
    # Step 2: Use ThreadPoolExecutor to run search_and_summarize in parallel for each sub-question
    # Step 3: Call synthesize_report to combine all findings into a final report
    # Step 4: Return the report
    """
    YOUR CODE HERE (~12 lines of code)
    """
    pass


# Run the multi-agent research
query = "What are the best resources to learn machine learning in 2025?"
output = plan_research(query)
# report = deep_research(query)
# print("=" * 60)
# print("FINAL REPORT")
# print("=" * 60)
# print(report)

['Simple factual queries:', 'What is the most popular programming language for machine learning in 2025?', 'Moderate topics:', 'What role do online courses play in machine learning education?', 'How do industry experts stay updated with the latest advancements in machine learning?', 'What are some key differences between supervised and unsupervised machine learning algorithms?', 'Complex topics needing multiple perspectives:', 'How do cultural and societal factors influence the adoption of machine learning technologies globally?', 'What is the impact of bias in machine learning models on real-world decision-making systems?', 'How can machine learning be used to address environmental sustainability challenges, considering multiple stakeholder perspectives?', 'In what ways can machine learning be integrated with other disciplines such as psychology or sociology to improve human-computer interaction?', 'To what extent do machine learning models reflect and perpetuate existing power struct

---
# 5- (Optional) Deploying the Deep Research App

In previous sections we built a deep research agent that runs inside a notebook. To make it available as a real application, we need three components:

1. **Model Server (Ollama)** — already running from Section 0, serves the LLM with an OpenAI-compatible API
2. **Application Server (FastAPI)** — wraps the research pipeline (planning, searching, synthesizing) as a REST API
3. **UI Client (Chainlit)** — a chat interface that sends queries to the application server and displays results

This is a common pattern for deploying LLM applications: separate the model serving from the application logic, so each layer can scale independently.

## 5.1: Dependencies

We need a few additional packages. Make sure those are installed in your environment:
- **chainlit**: Chat UI framework
- **httpx**: Async HTTP client for making API calls
- **fastapi** + **uvicorn**: Web server for the application API

## 5.2: Choose an inference engine and run its server

For production-ready applications, vLLM and SGLang are the most common options. They provide functionality similar to Ollama, but are more efficient and better optimized, so your LLM calls will be faster and support higher throughput. If you have a machine with a GPU, you can use vLLM or SGLang instead. In this project, we stick with Ollama because it is already set up, and we can run it locally without a GPU.

Make sure the `qwen2.5:3b-instruct` model is available, then start Ollama in a terminal:

```bash
ollama serve
```

## 5.3: Backend: Build the Application Server with FastAPI

The application server wraps our multi-agent deep research pipeline from Section 5 as a REST API. It connects to Ollama at `http://localhost:11434/v1` using the same OpenAI-compatible API format we have been using throughout this project.

Create a `deploy/` directory and a Python file `deploy/server.py`. In it, implement:

1. A **FastAPI app** that connects to Ollama using the OpenAI client
2. **Request/response models**: the request takes a `question` string; the response returns the `question`, a list of `sub_questions`, and the final `report`
3. The **three pipeline functions** from Section 5: `plan_research` (break query into sub-questions), `search_and_summarize` (search DuckDuckGo and summarize per sub-question), and `synthesize_report` (combine all findings into a markdown report)
4. A **`POST /research` endpoint** that orchestrates the full pipeline: plan → parallel search → synthesize

## 5.4: Frontend: Build the UI with Chainlit

[Chainlit](https://docs.chainlit.io/) gives us a polished chat interface with minimal code. Our app sends the user's message to the FastAPI server and displays the research report.

Create `deploy/app.py`. Implement a Chainlit chat interface that sends requests to your FastAPI server and displays the result.

## 5.5: Launch and Test

Make sure Ollama is running (`ollama serve`), then open **two terminals** and start each component:

**Terminal 1: Start application server:**

```bash
cd deploy && uvicorn server:app --host 0.0.0.0 --port 8001
```

**Terminal 2: Launch Chat UI:**
```bash
cd deploy && chainlit run app.py --port 8501
```

Open `http://localhost:8501` in your browser to use the chat interface.

## 🎉 Congratulations!

You have:
* Practiced various inference-time reasoning methods (CoT, self-consistency, sequential revision, ToT)
* Gained intuition about training reasoning models (STaR, ORM/PRM)
* Built a **deep-research agent** with tool calling and ReAct-style reasoning
* Implemented a **multi-agent system** with parallel research and report synthesis


👏 **Great job!** Take a moment to celebrate. The techniques you implemented here power many production agents and chatbots.